<a href="https://colab.research.google.com/github/henriquecarvalhodeandrade/Projeto_IA/blob/main/Projeto_IA_Fauna_BR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Configurando o Ambiente do Colab

###### OBS: Configurar o tipo da Runtime como GPU

In [ ]:
"""Configuração Inicial do Ambiente"""

# Instalação das dependências
!pip install ultralytics kaggle

import ultralytics
ultralytics.checks()

# Configurar Kaggle API
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 39.6/112.6 GB disk)


In [ ]:
"""Upload do kaggle.json"""

from google.colab import files

print("Envie o arquivo kaggle.json baixado do Kaggle:")
uploaded = files.upload()

# Nome do arquivo enviado
kaggle_file = list(uploaded.keys())[0]
kaggle_file = kaggle_file.replace(' ', '\\ ')

# Criar pasta .kaggle e mover arquivo
!mkdir -p ~/.kaggle
!cp {kaggle_file} ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

print("kaggle.json configurado com sucesso!")


!ls -l ~/.kaggle


Envie o arquivo kaggle.json baixado do Kaggle:


Saving kaggle.json to kaggle (2).json
/bin/bash: -c: line 1: syntax error near unexpected token `('
/bin/bash: -c: line 1: `cp kaggle\ (2).json ~/.kaggle/kaggle.json'
kaggle.json configurado com sucesso!
total 4
-rw------- 1 root root 72 Dec  7 19:35 kaggle.json


## Organizando o Dataset

In [ ]:
"""Download e Extração do Dataset"""

# Baixar dataset
!kaggle datasets download -d gabrielferrante/brazilian-road-animals-bra-dataset
!unzip -q brazilian-road-animals-bra-dataset.zip

# Conferir pastas
!ls Dataset


Dataset URL: https://www.kaggle.com/datasets/gabrielferrante/brazilian-road-animals-bra-dataset
License(s): CC0-1.0
brazilian-road-animals-bra-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)
replace .gitignore? [y]es, [n]o, [A]ll, [N]one, [r]ename: N
classes.txt	    convertYOLOtoXML.py  getDimensions.py  PASCAL_annotation
codigos_console.js  desktop.ini		 images		   requirements.txt
compactImages.py    download_images.py	 labels


In [ ]:
"""Preparação do Dataset para YOLOv8"""

import shutil
import os

# Criar estrutura vazia
for split in ["train", "val", "test"]:
    os.makedirs(f"bra-dataset/images/{split}", exist_ok=True)
    os.makedirs(f"bra-dataset/labels/{split}", exist_ok=True)

# Copiar imagens existentes
shutil.copytree("Dataset/images/train", "bra-dataset/images/train", dirs_exist_ok=True)
shutil.copytree("Dataset/images/val",   "bra-dataset/images/val",   dirs_exist_ok=True)

# Copiar labels
shutil.copytree("Dataset/labels/train", "bra-dataset/labels/train", dirs_exist_ok=True)
shutil.copytree("Dataset/labels/val",   "bra-dataset/labels/val",   dirs_exist_ok=True)

# Conferir estrutura final
!apt-get install tree -y
!tree bra-dataset -L 3


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tree is already the newest version (2.0.2-1).
0 upgraded, 0 newly installed, 0 to remove and 41 not upgraded.
bra-dataset
├── images
│   ├── test
│   ├── train
│   │   ├── anta00000000.jpg
│   │   ├── anta00000001.jpg
│   │   ├── anta00000002.jpg
│   │   ├── anta00000003.jpg
│   │   ├── anta00000004.jpg
│   │   ├── anta00000005.jpg
│   │   ├── anta00000006.jpg
│   │   ├── anta00000007.jpg
│   │   ├── anta00000008.jpg
│   │   ├── anta00000009.jpg
│   │   ├── anta00000010.jpg
│   │   ├── anta00000011.jpg
│   │   ├── anta00000014.jpg
│   │   ├── anta00000015.jpg
│   │   ├── anta00000016.jpg
│   │   ├── anta00000017.jpg
│   │   ├── anta00000018.jpg
│   │   ├── anta00000019.jpg
│   │   ├── anta00000020.jpg
│   │   ├── anta00000021.jpg
│   │   ├── anta00000022.jpg
│   │   ├── anta00000023.jpg
│   │   ├── anta00000024.jpg
│   │   ├── anta00000026.jpg
│   │   ├── anta00000027.jpg
│   │   ├── anta00

In [ ]:
"""Verificar estrutura do Dataset"""

# Adicione este código para verificar se os diretórios existem:
import os

print("Verificando estrutura do dataset:")
print(f"Train images exist: {os.path.exists('bra-dataset/images/train')}")
print(f"Train labels exist: {os.path.exists('bra-dataset/labels/train')}")
print(f"Val images exist: {os.path.exists('bra-dataset/images/val')}")
print(f"Val labels exist: {os.path.exists('bra-dataset/labels/val')}")

# Conte as imagens
train_imgs = len(os.listdir("bra-dataset/images/train")) if os.path.exists("bra-dataset/images/train") else 0
print(f"Total train images: {train_imgs}")

Verificando estrutura do dataset:
Train images exist: True
Train labels exist: True
Val images exist: True
Val labels exist: True
Total train images: 1460


In [ ]:
"""Verificar as classes do dataset"""

with open("Dataset/classes.txt", "r") as f:
    classes = f.read().strip().split('\n')
    print("Classes encontradas no dataset:")
    for i, cls in enumerate(classes):
        print(f"  {i}: {cls}")

Classes encontradas no dataset:
  0: Anta
  1: Jaguarundi
  2: LoboGuara
  3: OncaParda
  4: TamanduaBandeira


In [ ]:
"""YAML de Configuração do Dataset"""

yaml_content = """path: /content/bra-dataset
train: images/train
val: images/val
test: images/test

names:
  0: Anta
  1: Jaguarundi
  2: LoboGuara
  3: OncaParda
  4: TamanduaBandeira
"""

with open("bra-dataset.yaml", "w") as f:
    f.write(yaml_content)

print("Arquivo bra-dataset.yaml atualizado com as 5 classes brasileiras!")

Arquivo bra-dataset.yaml atualizado com as 5 classes brasileiras!


In [ ]:
"""Verificar arquivo .yaml"""

with open("bra-dataset.yaml", "r") as f:
    print("Conteúdo do bra-dataset.yaml:")
    print(f.read())

Conteúdo do bra-dataset.yaml:
path: /content/bra-dataset
train: images/train
val: images/val
test: images/test

names:
  0: Anta
  1: Jaguarundi
  2: LoboGuara
  3: OncaParda
  4: TamanduaBandeira



## Iniciando o Treinamento

In [ ]:
"""Treinamento Inicial"""
# (10 épocas para teste)
!yolo detect train data=bra-dataset.yaml model=yolov8n.pt epochs=10 imgsz=640 device=0


Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=bra-dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretraine

In [ ]:
"""Treinamento Final"""
# (50 épocas)

from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(
    data="bra-dataset.yaml",
    epochs=50,
    imgsz=640,
    workers=2
)


Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=bra-dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train3, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0, pretra

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e72ec49ade0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [ ]:
"""Avaliação e Teste do Modelo"""
# Verificar pesos gerados
!ls runs/detect/train/weights

# Testar com imagens de validação
model = YOLO("runs/detect/train/weights/best.pt")
model.predict(source="bra-dataset/images/val", save=True)

# Mostrar imagem anotada
from IPython.display import Image
Image(filename="runs/detect/predict/tamanduaBandeira00000377.jpg")

# Avaliar métricas do modelo
model.val()


best.pt  last.pt

image 1/363 /content/bra-dataset/images/val/anta00000345.jpg: 640x480 1 Anta, 59.2ms
image 2/363 /content/bra-dataset/images/val/anta00000368.jpg: 448x640 1 Anta, 42.9ms
image 3/363 /content/bra-dataset/images/val/anta00000391.jpg: 384x640 1 Anta, 38.6ms
image 4/363 /content/bra-dataset/images/val/anta00000396.jpg: 480x640 1 OncaParda, 36.9ms
image 5/363 /content/bra-dataset/images/val/anta00000397.jpg: 384x640 1 Anta, 6.7ms
image 6/363 /content/bra-dataset/images/val/anta00000398.jpg: 480x640 1 Anta, 6.7ms
image 7/363 /content/bra-dataset/images/val/anta00000399.jpg: 448x640 (no detections), 6.4ms
image 8/363 /content/bra-dataset/images/val/anta00000403.jpg: 416x640 3 Antas, 39.1ms
image 9/363 /content/bra-dataset/images/val/anta00000404.jpg: 448x640 1 Anta, 6.4ms
image 10/363 /content/bra-dataset/images/val/anta00000405.jpg: 384x640 1 Anta, 6.7ms
image 11/363 /content/bra-dataset/images/val/anta00000408.jpg: 384x640 1 Anta, 6.9ms
image 12/363 /content/bra-dataset/im

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e722a1bcd40>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

In [ ]:
"""Fine-Tuning (treino extra)"""

model = YOLO("runs/detect/train/weights/best.pt")

model.train(
    data="bra-dataset.yaml",
    epochs=100,
    imgsz=640,
    workers=2,
    lr0=0.0005
)


Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=bra-dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs/detect/train/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plo

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e72ec4ef770>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
        

## Testes com Imagens e Vídeos Externos ao Dataset

In [ ]:
"""Testar o Modelo com Imagem Local"""

# Fazer upload da imagem
from google.colab import files

uploaded = files.upload()
uploaded_file = list(uploaded.keys())[0]
uploaded_file

# Rodar predição na imagem
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/best.pt")

results = model.predict(
    source=uploaded_file,
    save=True
)

results

# Visualizar imagem detectada
from IPython.display import Image
import os

# pega o último diretório de predição
predict_dirs = sorted([d for d in os.listdir("runs/detect") if "predict" in d])
latest = f"runs/detect/{predict_dirs[-1]}"

# mostra primeira imagem
pred_img = os.listdir(latest)[0]
Image(filename=f"{latest}/{pred_img}")


IndexError: list index out of range

In [ ]:
"""Testar com Imagem da Internet"""

!wget https://ultralytics.com/images/bus.jpg -O test.jpg

model.predict(source="test.jpg", save=True)


--2025-12-07 21:25:53--  https://ultralytics.com/images/bus.jpg
Resolving ultralytics.com (ultralytics.com)... 198.202.211.1
Connecting to ultralytics.com (ultralytics.com)|198.202.211.1|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.ultralytics.com/images/bus.jpg [following]
--2025-12-07 21:25:53--  https://www.ultralytics.com/images/bus.jpg
Resolving www.ultralytics.com (www.ultralytics.com)... 198.202.211.1, 2620:cb:2000::1
Connecting to www.ultralytics.com (www.ultralytics.com)|198.202.211.1|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/ultralytics/assets/releases/download/v0.0.0/bus.jpg [following]
--2025-12-07 21:25:53--  https://github.com/ultralytics/assets/releases/download/v0.0.0/bus.jpg
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Loc

[ultralytics.engine.results.Results object with attributes:
 
 boxes: ultralytics.engine.results.Boxes object
 keypoints: None
 masks: None
 names: {0: 'Anta', 1: 'Jaguarundi', 2: 'LoboGuara', 3: 'OncaParda', 4: 'TamanduaBandeira'}
 obb: None
 orig_img: array([[[119, 146, 172],
         [121, 148, 174],
         [122, 152, 177],
         ...,
         [161, 171, 188],
         [160, 170, 187],
         [160, 170, 187]],
 
        [[120, 147, 173],
         [122, 149, 175],
         [123, 153, 178],
         ...,
         [161, 171, 188],
         [160, 170, 187],
         [160, 170, 187]],
 
        [[123, 150, 176],
         [124, 151, 177],
         [125, 155, 180],
         ...,
         [161, 171, 188],
         [160, 170, 187],
         [160, 170, 187]],
 
        ...,
 
        [[183, 182, 186],
         [179, 178, 182],
         [180, 179, 183],
         ...,
         [121, 111, 117],
         [113, 103, 109],
         [115, 105, 111]],
 
        [[165, 164, 168],
         [173,

In [ ]:
"""Testar com Vídeo Local"""
# Fazer upload do vídeo
from google.colab import files

uploaded = files.upload()
video_file = list(uploaded.keys())[0]
video_file

# Rodar predição no vídeo
model.predict(
    source=video_file,
    save=True,
    conf=0.25
)

# Baixar o vídeo processado
import os
from google.colab import files

predict_dirs = sorted([d for d in os.listdir("runs/detect") if "predict" in d])
latest = f"runs/detect/{predict_dirs[-1]}"

for f in os.listdir(latest):
    if f.endswith(".mp4"):
        files.download(f"{latest}/{f}")


IndexError: list index out of range

In [ ]:
"""Testar com Vídeo da Internet"""
!wget https://ultralytics.com/images/traffic.mp4 -O traffic.mp4

model.predict(
    source="traffic.mp4",
    save=True,
    conf=0.25
)


--2025-12-07 21:26:51--  https://ultralytics.com/images/traffic.mp4
Resolving ultralytics.com (ultralytics.com)... 198.202.211.1
Connecting to ultralytics.com (ultralytics.com)|198.202.211.1|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://www.ultralytics.com/images/traffic.mp4 [following]
--2025-12-07 21:26:52--  https://www.ultralytics.com/images/traffic.mp4
Resolving www.ultralytics.com (www.ultralytics.com)... 198.202.211.1, 2620:cb:2000::1
Connecting to www.ultralytics.com (www.ultralytics.com)|198.202.211.1|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/ultralytics/assets/releases/download/v0.0.0/traffic.mp4 [following]
--2025-12-07 21:26:52--  https://github.com/ultralytics/assets/releases/download/v0.0.0/traffic.mp4
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting respo

FileNotFoundError: Failed to open video /content/traffic.mp4

In [ ]:
"""Visualizar um Frame Detectado"""
from IPython.display import Image
import os

predict_dirs = sorted([d for d in os.listdir("runs/detect") if "predict" in d])
latest = f"runs/detect/{predict_dirs[-1]}"

# listar arquivos
os.listdir(latest)[:10]


['test.jpg']

In [ ]:
"""Visualizar Vídeo no Colab"""
from IPython.display import HTML
from base64 import b64encode

video_path = f"{latest}/video.mp4"

mp4 = open(video_path,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

HTML(f"""
<video width=640 controls>
    <source src="{data_url}" type="video/mp4">
</video>
""")


FileNotFoundError: [Errno 2] No such file or directory: 'runs/detect/predict2/video.mp4'